In [1]:
import requests
from bs4 import BeautifulSoup
import os
import time

### 1). Scraping the Data

In [2]:
def scrape_datasets(url, current_path):
    time.sleep(1) 
    
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
    except Exception as e:
        print(f"Error accessing {url}: {e}")
        return

    for link in soup.find_all('a'):
        href = link.get('href')
        
        if href in ['../', 'SUCCESS', '_SUCCESS'] or '?' in href:
            continue
            
        full_url = url + href
        
        if href.endswith('/'):
            # Only enter 'parquet' folders or general dataset folders
            # This avoids wasting time in the 'csv/' directories
            if 'csv' in href.lower():
                continue
                
            print(f"Entering folder: {href}")
            new_local_path = os.path.join(current_path, href.strip('/'))
            os.makedirs(new_local_path, exist_ok=True)
            scrape_datasets(full_url, new_local_path)
            
        elif href.startswith('part-'):
     
            if 'parquet' in url.lower():
                local_file_path = os.path.join(current_path, href)
                
                if not os.path.exists(local_file_path):
                    print(f"Downloading Parquet: {href}")
                    try:
                        with requests.get(full_url, stream=True) as r:
                            r.raise_for_status()
                            with open(local_file_path, 'wb') as f:
                                for chunk in r.iter_content(chunk_size=8192):
                                    f.write(chunk)
                    except Exception as e:
                        print(f"Failed to download {href}: {e}")

In [12]:
BASE_URL = "https://datasets.uwf.edu/data/"
DOWNLOAD_DIR = "/mnt/c/SJSU/Spring 2026/DATA 298A/02 Sample Data/UWF_Data"

if not os.path.exists(DOWNLOAD_DIR):
    os.makedirs(DOWNLOAD_DIR)
    
scrape_datasets(BASE_URL, DOWNLOAD_DIR)

Entering folder: UWF-ZeekData22/
Entering folder: parquet/
Entering folder: 2021-12-12%20-%202021-12-19/
Entering folder: 2021-12-19%20-%202021-12-26/
Entering folder: 2021-12-26%20-%202022-01-02/
Entering folder: 2022-01-02%20-%202022-01-09/
Entering folder: 2022-01-09%20-%202022-01-16/
Entering folder: 2022-01-16%20-%202022-01-23/
Entering folder: 2022-02-06%20-%202022-02-13/
Entering folder: 2022-02-13%20-%202022-02-20/
Entering folder: _parquet_file_metrics_/
Entering folder: UWF-ZeekData24/
Entering folder: parquet/
Entering folder: 2024-02-25%20-%202024-03-03/
Entering folder: 2024-03-03%20-%202024-03-10/
Entering folder: 2024-03-10%20-%202024-03-17/
Entering folder: 2024-03-17%20-%202024-03-24/
Entering folder: 2024-03-24%20-%202024-03-31/
Entering folder: 2024-10-27%20-%202024-11-03/
Entering folder: 2024-11-03%20-%202024-11-10/
Entering folder: _parquet_binary_metrics_/
Entering folder: _parquet_cve_metrics_/
Entering folder: _parquet_tactic_metrics_/
Entering folder: _parquet

### 2). Combining Data

In [15]:
import pandas as pd
import glob
import os

In [20]:
# !pip install fastparquet

In [21]:
BASE_DIR = "/mnt/c/SJSU/Spring 2026/DATA 298A/02 Sample Data/UWF_Data"

def inspect_parquet_files(root_dir):
    print(f"{'Folder Name':<40} | {'Shape'}")
    print("-" * 60)
    
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if "part-" in file and file.endswith(".parquet"):
                file_path = os.path.join(root, file)
                
                try:
                    # Switch to fastparquet to avoid the pyarrow extension bug
                    df = pd.read_parquet(file_path, engine='fastparquet')
                    
                    folder_name = os.path.basename(root)
                    print(f"{folder_name[:38]:<40} | {df.shape}")
                    
                except Exception as e:
                    print(f"Error reading {file}: {e}")

if __name__ == "__main__":
    if os.path.exists(BASE_DIR):
        inspect_parquet_files(BASE_DIR)
    else:
        print(f"Path not found: {BASE_DIR}")

Folder Name                              | Shape
------------------------------------------------------------
2021-12-12%20-%202021-12-19              | (428724, 23)
2021-12-19%20-%202021-12-26              | (1757069, 23)
2021-12-26%20-%202022-01-02              | (1757998, 23)
2022-01-02%20-%202022-01-09              | (2403853, 23)
2022-01-09%20-%202022-01-16              | (2579855, 23)
2022-01-16%20-%202022-01-23              | (354100, 23)
2022-02-06%20-%202022-02-13              | (9280806, 23)
2022-02-13%20-%202022-02-20              | (63, 23)
2024-02-25%20-%202024-03-03              | (255443, 26)
2024-03-03%20-%202024-03-10              | (50444, 26)
2024-03-10%20-%202024-03-17              | (167578, 26)
2024-03-17%20-%202024-03-24              | (378821, 26)
2024-03-24%20-%202024-03-31              | (106362, 26)
2024-10-27%20-%202024-11-03              | (454846, 26)
2024-11-03%20-%202024-11-10              | (503263, 26)
2021-12-12%20-%202021-12-19              | (20745,

**~27 million data points**

In [23]:
BASE_DIR = "/mnt/c/SJSU/Spring 2026/DATA 298A/02 Sample Data/UWF_Data"

def inspect_parquet_files(root_dir):
    print(f"{'Folder Name':<40} | {'Column Name'}")
    print("-" * 60)
    
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if "part-" in file and file.endswith(".parquet"):
                file_path = os.path.join(root, file)
                
                try:
                    # Switch to fastparquet to avoid the pyarrow extension bug
                    df = pd.read_parquet(file_path, engine='fastparquet')
                    
                    folder_name = os.path.basename(root)
                    print(f"{folder_name[:38]:<40} | {df.columns}")
                    
                except Exception as e:
                    print(f"Error reading {file}: {e}")

if __name__ == "__main__":
    if os.path.exists(BASE_DIR):
        inspect_parquet_files(BASE_DIR)
    else:
        print(f"Path not found: {BASE_DIR}")

Folder Name                              | Column Name
------------------------------------------------------------
2021-12-12%20-%202021-12-19              | Index(['resp_pkts', 'service', 'orig_ip_bytes', 'local_resp', 'missed_bytes',
       'proto', 'duration', 'conn_state', 'dest_ip_zeek', 'orig_pkts',
       'community_id', 'resp_ip_bytes', 'dest_port_zeek', 'orig_bytes',
       'local_orig', 'datetime', 'history', 'resp_bytes', 'uid',
       'src_port_zeek', 'ts', 'src_ip_zeek', 'label_tactic'],
      dtype='str')
2021-12-19%20-%202021-12-26              | Index(['resp_pkts', 'service', 'orig_ip_bytes', 'local_resp', 'missed_bytes',
       'proto', 'duration', 'conn_state', 'dest_ip_zeek', 'orig_pkts',
       'community_id', 'resp_ip_bytes', 'dest_port_zeek', 'orig_bytes',
       'local_orig', 'datetime', 'history', 'resp_bytes', 'uid',
       'src_port_zeek', 'ts', 'src_ip_zeek', 'label_tactic'],
      dtype='str')
2021-12-26%20-%202022-01-02              | Index(['resp_pkts', '

**Number of columns are not consistent over the years : ranges from 23 to 27 : later years columns for MITRE and CVE related info has been added**

**'label_technique', 'label_cve', 'label_binary', 'vlan' columns are missing in initial years**

In [25]:
from fastparquet import write

In [27]:
BASE_DIR = "/mnt/c/SJSU/Spring 2026/DATA 298A/02 Sample Data/UWF_Data"
OUTPUT_FILE = os.path.join(BASE_DIR, "combined_uwf_dataset.parquet")

def merge_parquets_to_disk(root_dir, output_path):
    # 1. Update pattern to ONLY catch .parquet files (skipping the 20 CSV metrics)
    search_pattern = os.path.join(root_dir, "**", "parquet", "**", "part-*.parquet")
    files = glob.glob(search_pattern, recursive=True)
    
    if not files:
        print("No parquet files found. Check your directory paths.")
        return

    if os.path.exists(output_path):
        os.remove(output_path)
        print(f"Removed existing {output_path}")

    # Define the 4 columns that are missing in older datasets
    new_cols = ['label_technique', 'label_cve', 'label_binary', 'vlan']
    
    first_file = True
    print(f"Starting direct merge of {len(files)} data files...")

    for i, file_path in enumerate(files):
        try:
            # Load chunk
            df = pd.read_parquet(file_path, engine='fastparquet')
            
            # Add Source Metadata
            folder_name = os.path.basename(os.path.dirname(file_path))
            df['source_period'] = folder_name
            
            # 2. Normalize Schema: Ensure all 27 columns exist before writing
            for col in new_cols:
                if col not in df.columns:
                    df[col] = 'unknown'
                else:
                    # Force to string to prevent the 'bad argument type' error
                    df[col] = df[col].astype(str).replace('nan', 'unknown')

            # 3. Write/Append
            if first_file:
                # The first write sets the schema for the entire file
                write(output_path, df, compression='SNAPPY')
                first_file = False
                print(f"Initialized Master File with schema from: {folder_name}")
            else:
                # Append subsequent files
                write(output_path, df, append=True, compression='SNAPPY')
            
            if i % 10 == 0:
                print(f"Progress: {i}/{len(files)} files processed.")

        except Exception as e:
            print(f"Error on {file_path}: {e}")

    print(f"\nSuccess! Combined dataset with 27M+ rows saved to: {output_path}")

if __name__ == "__main__":
    merge_parquets_to_disk(BASE_DIR, OUTPUT_FILE)

Starting direct merge of 50 data files...
Initialized Master File with schema from: 2021-12-12%20-%202021-12-19
Progress: 0/50 files processed.
Progress: 10/50 files processed.
Progress: 20/50 files processed.
Progress: 30/50 files processed.
Progress: 40/50 files processed.

Success! Combined dataset with 27M+ rows saved to: /mnt/c/SJSU/Spring 2026/DATA 298A/02 Sample Data/UWF_Data/combined_uwf_dataset.parquet


In [36]:
import pyarrow.parquet as pq
parquet_file = pq.ParquetFile(OUTPUT_FILE)

# Get Shape
rows = parquet_file.metadata.num_rows
cols_count = parquet_file.metadata.num_columns
print(f"Shape: ({rows}, {cols_count})")

# Get Column Names in Order
column_names = parquet_file.schema.names
print("\nColumn Names in Parquet Order:")
for i, name in enumerate(column_names):
    print(f"{i}: {name}")

Shape: (27111594, 28)

Column Names in Parquet Order:
0: resp_pkts
1: service
2: orig_ip_bytes
3: local_resp
4: missed_bytes
5: proto
6: duration
7: conn_state
8: dest_ip_zeek
9: orig_pkts
10: community_id
11: resp_ip_bytes
12: dest_port_zeek
13: orig_bytes
14: local_orig
15: datetime
16: history
17: resp_bytes
18: uid
19: src_port_zeek
20: ts
21: src_ip_zeek
22: label_tactic
23: source_period
24: label_technique
25: label_cve
26: label_binary
27: vlan


### 4). EDA

In [28]:
# !pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 30.7 MB/s eta 0:00:0000:0100:01


In [1]:
import duckdb

In [2]:
con = duckdb.connect()

In [4]:
import os

In [5]:
BASE_DIR = "/mnt/c/SJSU/Spring 2026/DATA 298A/02 Sample Data/UWF_Data"
PARQUET_FILE = "/mnt/c/SJSU/Spring 2026/DATA 298A/02 Sample Data/UWF_Data/combined_uwf_dataset.parquet"
DB_FILE = os.path.join(BASE_DIR, "uwf_cybersecurity.db")

In [ ]:
# using created .db to load table
DB_FILE = "/mnt/c/SJSU/Spring 2026/DATA 298A/02 Sample Data/UWF_Data/uwf_cybersecurity.db"
con = duckdb.connect(DB_FILE)

In [7]:
# # Creating table with schema from paraquet file
# con.execute("""
# CREATE TABLE network_logs (
#     resp_pkts BIGINT,           -- 0
#     service VARCHAR,            -- 1
#     orig_ip_bytes BIGINT,       -- 2
#     local_resp BOOLEAN,         -- 3
#     missed_bytes BIGINT,        -- 4
#     proto VARCHAR,              -- 5
#     duration DOUBLE,            -- 6
#     conn_state VARCHAR,         -- 7
#     dest_ip_zeek VARCHAR,       -- 8
#     orig_pkts BIGINT,           -- 9
#     community_id VARCHAR,       -- 10
#     resp_ip_bytes BIGINT,       -- 11
#     dest_port_zeek INTEGER,     -- 12
#     orig_bytes BIGINT,          -- 13
#     local_orig BOOLEAN,         -- 14
#     datetime TIMESTAMP,         -- 15 
#     history VARCHAR,            -- 16
#     resp_bytes BIGINT,          -- 17
#     uid VARCHAR,                -- 18
#     src_port_zeek INTEGER,      -- 19
#     ts DOUBLE,                  -- 20
#     src_ip_zeek VARCHAR,        -- 21
#     label_tactic VARCHAR,       -- 22
#     source_period VARCHAR,      -- 23
#     label_technique VARCHAR,    -- 24
#     label_cve VARCHAR,          -- 25
#     label_binary VARCHAR,       -- 26
#     vlan VARCHAR                -- 27
# );
# """)
# print("Table schema created successfully.")

Table schema created successfully.


In [8]:
# con.execute(f"INSERT INTO network_logs SELECT * FROM read_parquet('{PARQUET_FILE}')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
# First few rows
con.execute("SELECT * FROM network_logs LIMIT 5").df()

,resp_pkts,service,orig_ip_bytes,local_resp,missed_bytes,proto,duration,conn_state,dest_ip_zeek,orig_pkts,...,uid,src_port_zeek,ts,src_ip_zeek,label_tactic,source_period,label_technique,label_cve,label_binary,vlan
0,1689365,NaN,141906660,False,0,icmp,1.729902e+06,OTH,143.88.0.1,1689365,...,Caxllt2aEzXOEZnpii,8,1.639746e+09,143.88.255.50,none,2021-12-12%20-%202021-12-19,unknown,unknown,unknown,unknown
1,0,NaN,462247452,False,0,icmp,1.730391e+06,OTH,143.88.255.10,4576716,...,CEx0rX1Yic8axI4sw1,3,1.639746e+09,143.88.255.1,none,2021-12-12%20-%202021-12-19,unknown,unknown,unknown,unknown
2,0,NaN,22008480,False,0,icmp,1.730410e+06,OTH,ff02::1,229255,...,Cpn5bT1QW9MPFXikn4,134,1.639746e+09,fe80::250:56ff:fe9e:da15,none,2021-12-12%20-%202021-12-19,unknown,unknown,unknown,unknown
3,0,NaN,34970488,False,0,icmp,1.730411e+06,OTH,ff02::16,460138,...,Ceo6B73UmSZVe8vsD4,143,1.639746e+09,fe80::250:56ff:fe9e:ef90,none,2021-12-12%20-%202021-12-19,unknown,unknown,unknown,unknown
4,0,dhcp,66459795,False,0,udp,1.731504e+06,S0,255.255.255.255,202621,...,CTCPKc2faq2rLYJA4f,68,1.639746e+09,0.0.0.0,none,2021-12-12%20-%202021-12-19,unknown,unknown,unknown,unknown


In [12]:
# shape of data

con.execute("SELECT COUNT(*) FROM network_logs").fetchone()[0]

27111594